In [2]:
# %% [markdown]
# # Clustering exploratorio de la estructura de embeddings por palabra (parte 2 de 2)
#
# **Alcance del análisis, léelo antes de interpretar resultados:** se probó
# quitar componentes principales dominantes (whitening) y medir silhouette
# score para intentar distinguir separaciones "reales" (sentido léxico) de
# separaciones espurias (ruido de registro/tema/longitud del tweet) en
# palabras de control conocidas. **No se encontró ninguna combinación de
# parámetros que lograra esa distinción** — las distribuciones de silhouette
# de splits en palabras monosémicas y polisémicas de control se superponen
# casi por completo (ver calibración, sección 3). Por lo tanto, este notebook
# ya NO reporta "polisemia" como verdad determinada — reporta **estructura de
# clusters observada en los embeddings de cada modelo** (cuántos grupos
# separables produce, qué tan compactos son), sin asumir que esa estructura
# corresponda a sentidos léxicos reales. Las columnas de salida se llaman
# `k_clusters` y `presenta_multiples_clusters` (no `k_sentidos`/`es_polisemica`)
# a propósito, para no sobre-afirmar lo que el método puede validar.
#
# Lee los `.npz` guardados por `extraer_embeddings.py` — no necesita GPU ni
# `transformers`, solo numpy/pandas/umap/hdbscan/sklearn.
#
# Requiere: umap-learn, hdbscan, scikit-learn, pandas, numpy
# Ejecuta: pip install umap-learn hdbscan scikit-learn pandas numpy --break-system-packages

# %%
import os
import csv
import unicodedata
import numpy as np
import pandas as pd
import umap
import hdbscan
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from collections import defaultdict

C:\Users\msubr\Anaconda3\envs\gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# %% [markdown]
# ## Configuración

# %%
MODELOS = ["mex_state", "mex_large", "beto", "bertin", "robertuito"]
CARPETA_EMBEDDINGS = "embeddings"

UMAP_N_COMPONENTS = 12
HDBSCAN_MIN_SAMPLES = 3
MIN_OCURRENCIAS_PARA_CLUSTER = 8   # palabras con menos ocurrencias no se agrupan

In [4]:
# %% [markdown]
# ## 0. Cargar embeddings guardados
#
# Lee el `.npz` de un modelo y devuelve `(embeddings, metadata)`, donde
# `metadata` es una lista de dicts con al menos la clave `palabra` por fila
# (más `id_tweet`/`region` si el `.npz` las incluye). Si guardaste el tuyo con
# `extraer_embeddings.py` las claves son `embeddings`/`words`/`id_tweets` — si
# usaste tu propio notebook con otras claves (ej. `region` incluida), esta
# función las toma tal cual vengan, solo mapea `words` → `palabra`.

# %%
def cargar_embeddings(nombre_modelo, carpeta=CARPETA_EMBEDDINGS):
    ruta = os.path.join(carpeta, f"{nombre_modelo}_embeddings.npz")
    data = np.load(ruta, allow_pickle=True)
    emb = data["embeddings"]
    words = data["words"]
    id_tweets = data["id_tweets"]
    return emb, words, id_tweets

# Verifica una vez qué claves trae tu .npz antes de seguir:
# _, _w = cargar_embeddings(MODELOS[0])
# print(_w[0])

In [86]:
# %% [markdown]
# ## 1. Clustering exploratorio por palabra (HDBSCAN)
#
# **`min_cluster_size` adaptativo, no fijo.** Un valor fijo (ej. 3) es razonable
# para una palabra con 20 ocurrencias, pero para una con 1,000+ ocurrencias es
# absurdamente permisivo — HDBSCAN encuentra decenas de micro-agrupaciones que
# son ruido estadístico (esto se confirmó con la corrida completa: 0% de
# palabras con k=1 en los 5 modelos, y k máximo de hasta 281). Este ajuste
# sigue siendo válido y necesario independientemente de la limitación de
# fondo descrita arriba — evita resultados absurdos, aunque no garantiza que
# el k resultante sea "sentidos reales".
#
# **`quitar_componentes_dominantes` (whitening) queda disponible pero SIN
# evidencia de que resuelva el problema de fondo.** Se probó como corrección
# contra anisotropía de embeddings de BERT, pero la calibración (sección 3)
# mostró que ni el número de clusters ni el silhouette score distinguen
# separaciones reales de espurias, con o sin esta corrección. Se deja
# implementada por si se retoma en el futuro con otro enfoque de validación,
# pero el valor final usado en la orquestación (sección 4) es 0.

# %%
MCS_FRACCION = 0.05   # min_cluster_size ≈ 5% de las ocurrencias de la palabra
MCS_MINIMO = 5
MCS_MAXIMO = 50

SEMILLAS_ESTABILIDAD = [0, 1, 2, 3, 4]  # 5 semillas -- balance entre costo computacional y robustez

def calcular_min_cluster_size(n, fraccion=MCS_FRACCION, minimo=MCS_MINIMO, maximo=MCS_MAXIMO):
    return int(np.clip(round(fraccion * n), minimo, maximo))


def quitar_componentes_dominantes(X, n_componentes=1):
    """Centra y quita las primeras n_componentes principales de las ocurrencias
    de UNA palabra. La dirección de mayor varianza en embeddings contextuales
    de BERT suele ser genérica (anisotropía: registro, longitud, tema) y no
    semántica — quitarla antes de clusterizar evita que HDBSCAN la confunda
    con sentidos distintos."""
    n_comp = min(n_componentes, X.shape[0] - 1, X.shape[1])
    media = X.mean(axis=0)
    X_centrado = X - media
    if n_comp < 1:
        return X_centrado
    pca = PCA(n_components=n_comp)
    pca.fit(X_centrado)
    proyeccion = pca.transform(X_centrado) @ pca.components_
    return X_centrado - proyeccion


def _hdbscan_labels(embeddings, min_cluster_size, min_samples, n_componentes_quitar=0, semilla=42):
    """Corre UMAP+HDBSCAN sobre los embeddings de una palabra y devuelve
    (labels, silhouette). labels: array de labels (uno por ocurrencia, -1 =
    ruido). silhouette: score de separación entre clusters (None si k<2 o muy
    pocos puntos no-ruido). Si n_componentes_quitar>0, primero centra y quita
    esa(s) componente(s) principal(es) dominante(s) (ver
    quitar_componentes_dominantes)."""
    X = np.vstack(embeddings)
    if n_componentes_quitar > 0:
        X = quitar_componentes_dominantes(X, n_componentes_quitar)
    n = len(embeddings)
    n_components = min(UMAP_N_COMPONENTS, n - 2)
    n_neighbors = min(15, n - 2)
    # n_jobs=1 es NECESARIO además de random_state -- UMAP paraleliza partes del
    # cálculo (búsqueda de vecinos, optimización) por defecto, y eso puede dar
    # resultados distintos entre corridas incluso con semilla fija. Sin esto,
    # el número de clusters puede variar (se detectó variación de 2 a 8 para la
    # misma palabra/modelo entre corridas) -- invalida cualquier lectura manual
    # de resultados si no se fuerza un solo hilo.
    reducer = umap.UMAP(n_components=n_components, n_neighbors=n_neighbors, random_state=semilla, n_jobs=1)
    X_red = reducer.fit_transform(X)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples)
    labels = clusterer.fit_predict(X_red)
 
    k = len(set(labels) - {-1})
    mask = labels != -1
    if k >= 2 and mask.sum() >= k + 1:
        silhouette = silhouette_score(X_red[mask], labels[mask])
    else:
        silhouette = None
    return labels, silhouette
 
 
def _hdbscan_multi_semilla(embeddings, min_cluster_size, min_samples, n_componentes_quitar=0,
                             semillas=SEMILLAS_ESTABILIDAD):
    """Corre _hdbscan_labels con varias semillas (ver hallazgo: reproducibilidad
    con una sola semilla NO implica que el resultado sea estable -- 'elefante'
    dio k=0 en el 100% de 15 semillas y aun así era incorrecto por otro motivo,
    pero otras palabras como 'internet'/'celular' variaron de k=2 a k=9 según
    la semilla). Devuelve un dict con k_clusters=moda, pct_semillas_moda
    (% de semillas que coincidieron con la moda), y pct_ruido/silhouette
    promediados sobre las semillas que dieron esa moda (más representativo
    que tomar una sola semilla al azar)."""
    resultados_semilla = []
    for semilla in semillas:
        labels, silhouette = _hdbscan_labels(embeddings, min_cluster_size, min_samples,
                                               n_componentes_quitar=n_componentes_quitar, semilla=semilla)
        k = len(set(labels) - {-1})
        pct_ruido = (labels == -1).mean() * 100
        resultados_semilla.append({"k": k, "pct_ruido": pct_ruido, "silhouette": silhouette})
 
    ks = [r["k"] for r in resultados_semilla]
    valores, conteos = np.unique(ks, return_counts=True)
    k_moda = int(valores[np.argmax(conteos)])
    pct_semillas_moda = conteos.max() / len(ks) * 100
 
    en_moda = [r for r in resultados_semilla if r["k"] == k_moda]
    pct_ruido_prom = float(np.mean([r["pct_ruido"] for r in en_moda]))
    silhouettes_validos = [r["silhouette"] for r in en_moda if r["silhouette"] is not None]
    silhouette_prom = float(np.mean(silhouettes_validos)) if silhouettes_validos else None
 
    return {
        "k_clusters": k_moda, "pct_semillas_moda": pct_semillas_moda,
        "pct_ruido": pct_ruido_prom, "silhouette": silhouette_prom,
    }
 
 
def clusterizar_palabra(embeddings, min_cluster_size=None, min_samples=None, n_componentes_quitar=0,
                          semillas=SEMILLAS_ESTABILIDAD):
    """Dado un array de embeddings (n_ocurrencias, dim) para UNA palabra en UN modelo,
    devuelve (k_clusters, pct_ruido, min_cluster_size_usado, silhouette,
    pct_semillas_moda). Corre con varias semillas (ver _hdbscan_multi_semilla)
    -- reproducibilidad con una sola semilla no implica que el resultado sea
    estable. Si no se pasa min_cluster_size explícito, se calcula de forma
    adaptativa según `n` (ver calcular_min_cluster_size)."""
    n = len(embeddings)
    if n < MIN_OCURRENCIAS_PARA_CLUSTER:
        return None, None, None, None, None  # no evaluada, muy pocas ocurrencias
 
    mcs = min_cluster_size if min_cluster_size is not None else calcular_min_cluster_size(n)
    ms = min_samples if min_samples is not None else HDBSCAN_MIN_SAMPLES
 
    resultado = _hdbscan_multi_semilla(embeddings, mcs, ms, n_componentes_quitar=n_componentes_quitar, semillas=semillas)
    return resultado["k_clusters"], resultado["pct_ruido"], mcs, resultado["silhouette"], resultado["pct_semillas_moda"]
 
 
def inspeccionar_ocurrencias(palabra, embeddings_dict, tweets_dict, min_cluster_size=None, min_samples=None,
                               n_componentes_quitar=0, semilla=42):
    """Devuelve un DataFrame (id_tweet, cluster_id) para revisar caso por caso
    a qué tweet corresponde cada ocurrencia y en qué cluster cayó (cluster,
    no "sentido" — ver limitación en el encabezado del notebook). Usa UNA sola
    semilla (para tener labels concretos que leer) -- si quieres confirmar que
    el k de esa semilla es representativo (la moda), corre antes
    medir_estabilidad.py para esa palabra."""
    embs = embeddings_dict[palabra]
    tids = tweets_dict[palabra]
    mcs = min_cluster_size if min_cluster_size is not None else calcular_min_cluster_size(len(embs))
    ms = min_samples if min_samples is not None else HDBSCAN_MIN_SAMPLES
    labels, _ = _hdbscan_labels(embs, mcs, ms, n_componentes_quitar=n_componentes_quitar, semilla=semilla)
    return pd.DataFrame({"id_tweet": tids, "cluster_id": labels})
 

In [6]:
# %% [markdown]
# ## 2. Filtro de palabras funcionales (stopwords)
#
# Palabras funcionales de alta frecuencia (preposiciones, artículos...) se
# fragmentan en decenas de micro-clusters espurios si no se excluyen — no
# tiene sentido preguntarse si son "polisémicas" en el sentido léxico que
# interesa aquí. Se excluyen del clustering (quedan registradas con
# `excluida_stopword=True` para trazabilidad, no se eliminan silenciosamente).

# %%
try:
    import nltk
    from nltk.corpus import stopwords as nltk_stopwords
    nltk.download("stopwords", quiet=True)
    STOPWORDS_ES = set(nltk_stopwords.words("spanish"))
except Exception:
    # Respaldo si no hay conexión para nltk.download
    STOPWORDS_ES = {
        "de", "la", "que", "el", "en", "y", "a", "los", "del", "se", "las", "por", "un",
        "para", "con", "no", "una", "su", "al", "lo", "como", "más", "pero", "sus", "le",
        "ya", "o", "este", "sí", "porque", "esta", "entre", "cuando", "muy", "sin", "sobre",
        "también", "me", "hasta", "hay", "donde", "quien", "desde", "todo", "nos", "durante",
        "todos", "uno", "les", "ni", "contra", "otros", "ese", "eso", "ante", "ellos", "e",
        "esto", "mí", "antes", "algunos", "qué", "unos", "yo", "otro", "otras", "otra", "él",
        "tanto", "esa", "estos", "mucho", "quienes", "nada", "muchos", "cual", "poco", "ella",
        "estar", "estas", "algunas", "algo", "nosotros", "mi", "mis", "tú", "te", "ti", "tu",
        "tus", "ellas", "nosotras", "vosotros", "vosotras", "os", "esos", "esas", "soy",
        "eres", "es", "somos", "sois", "son", "era", "eras", "fui",
    }
print(f"Stopwords cargadas: {len(STOPWORDS_ES)}")

Stopwords cargadas: 106


In [7]:
# %% [markdown]
# ## 3. Calibración/diagnóstico exploratorio — correr ANTES del pipeline completo
#
# **Resultado ya conocido: ninguna combinación aquí "valida" polisemia real**
# (ver limitación en el encabezado). Esta sección se deja como diagnóstico de
# comportamiento del pipeline (qué tan sensible es a los parámetros, cómo se
# comparan los 5 modelos entre sí) — no como búsqueda de la config "correcta".
# Carga las palabras de calibración directamente de los `.npz` ya guardados.

# %%
PALABRAS_CALIBRACION_POLISEMICAS = {"cambio", "elefante", "celular", "planta", "sierra", "cola", "papa", "vela", }
PALABRAS_CALIBRACION_MONOSEMICAS = {"bicicleta", "computadora", "internet", "anillo",
                                      "aeropuerto", "estudiante", "diciembre", "botella"}
PALABRAS_CALIBRACION = PALABRAS_CALIBRACION_POLISEMICAS | PALABRAS_CALIBRACION_MONOSEMICAS

embeddings_calib_por_modelo = {}
tweets_calib_por_modelo = {}

for nombre_modelo in MODELOS:
    emb, w, id_tweets= cargar_embeddings(nombre_modelo)
    emb /= np.linalg.norm(emb, axis=1).reshape(-1, 1)

    # normaliza a NFC -- "ñ" puede venir como un solo carácter o como "n" + tilde
    # combinada; se ven idénticas pero no son el mismo string en Python
    palabras_arr = np.array([unicodedata.normalize("NFC", str(d)) for d in w])
    tweets_arr = np.array(id_tweets)

    emb_dict, tw_dict = defaultdict(list), defaultdict(list)
    for palabra in PALABRAS_CALIBRACION:
        palabra_nfc = unicodedata.normalize("NFC", palabra)
        idx = np.where(palabras_arr == palabra_nfc)[0]
        if len(idx) == 0:
            print(f"  [{nombre_modelo}] '{palabra}' no está en el .npz guardado — sáltala o revisa el nombre")
            continue
        emb_dict[palabra] = list(emb[idx])
        tw_dict[palabra] = list(tweets_arr[idx])
    embeddings_calib_por_modelo[nombre_modelo] = emb_dict
    tweets_calib_por_modelo[nombre_modelo] = tw_dict

resultados_calibracion = []
CONFIGS_CALIBRACION = ["adaptativo", 3, 5, 10]
COMPONENTES_A_PROBAR = [0, 1, 2, 3, 5]

for nombre_modelo in MODELOS:
    for palabra, embs in embeddings_calib_por_modelo[nombre_modelo].items():
        tipo_esperado = "polisémica" if palabra in PALABRAS_CALIBRACION_POLISEMICAS else "monosémica"
        for config in CONFIGS_CALIBRACION:
            mcs_explicito = None if config == "adaptativo" else config
            for n_comp in COMPONENTES_A_PROBAR:
                k, pct_ruido, mcs_usado, silhouette = clusterizar_palabra(
                    embs, min_cluster_size=mcs_explicito, min_samples=HDBSCAN_MIN_SAMPLES,
                    n_componentes_quitar=n_comp,
                )
                resultados_calibracion.append({
                    "modelo": nombre_modelo, "palabra": palabra, "tipo_esperado": tipo_esperado,
                    "n_ocurrencias": len(embs), "config": config, "min_cluster_size_usado": mcs_usado,
                    "n_componentes_quitar": n_comp, "k_clusters": k, "pct_ruido": pct_ruido,
                    "silhouette": silhouette,
                })

calibracion_df = pd.DataFrame(resultados_calibracion)

# En vez de asumir un umbral de silhouette, comparamos la distribución real:
# separaciones "reales" (polisémicas con k>=2) vs. "espurias" (monosémicas con
# k>=2) -- si hay una brecha clara, ahí está el corte a usar; si se solapan,
# el silhouette tampoco distingue y hay que pensar en otra cosa.
splits_reales = calibracion_df[(calibracion_df["tipo_esperado"] == "polisémica") & (calibracion_df["k_clusters"] >= 2)]
splits_espurios = calibracion_df[(calibracion_df["tipo_esperado"] == "monosémica") & (calibracion_df["k_clusters"] >= 2)]

print("=== Distribución de silhouette en splits de polisémicas (se espera que sean reales) ===")
print(splits_reales.groupby("n_componentes_quitar")["silhouette"].describe())

print("\n=== Distribución de silhouette en splits de monosémicas (se espera que sean espurios) ===")
print(splits_espurios.groupby("n_componentes_quitar")["silhouette"].describe())

=== Distribución de silhouette en splits de polisémicas (se espera que sean reales) ===
                      count      mean       std       min       25%       50%  \
n_componentes_quitar                                                            
0                     134.0  0.640186  0.170381  0.230554  0.523561  0.586176   
1                     125.0  0.479539  0.162465  0.153504  0.378327  0.446640   
2                     114.0  0.439041  0.164128  0.077412  0.330937  0.404200   
3                     118.0  0.390651  0.133590  0.204332  0.308388  0.360427   
5                     107.0  0.299072  0.112954  0.028035  0.224463  0.285920   

                           75%       max  
n_componentes_quitar                      
0                     0.792228  0.933989  
1                     0.558492  0.931464  
2                     0.493253  0.896022  
3                     0.429866  0.785950  
5                     0.367013  0.660032  

=== Distribución de silhouette en splits d

In [ ]:
N_SEMILLAS = 15
SEMILLAS = list(range(N_SEMILLAS))


def _hdbscan_labels_con_semilla(embeddings, min_cluster_size, min_samples, semilla, n_componentes_quitar=0):
    X = np.vstack(embeddings)
    if n_componentes_quitar > 0:
        X = quitar_componentes_dominantes(X, n_componentes_quitar)  # reutiliza la función ya definida en clusterizar_sentidos.py
    n = len(embeddings)
    n_components = min(UMAP_N_COMPONENTS, n - 2)
    n_neighbors = min(15, n - 2)
    reducer = umap.UMAP(n_components=n_components, n_neighbors=n_neighbors, random_state=semilla, n_jobs=1)
    X_red = reducer.fit_transform(X)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples)
    return clusterer.fit_predict(X_red)


def medir_estabilidad(modelo, palabra, embeddings, min_cluster_size=None, min_samples=None, n_componentes_quitar=0, semillas=SEMILLAS):
    """Corre el clustering de una palabra con varias semillas y devuelve un
    resumen: lista de k por semilla, moda, y qué % de semillas coinciden con
    la moda (medida de estabilidad, no de "verdad")."""
    n = len(embeddings)
    mcs = min_cluster_size if min_cluster_size is not None else calcular_min_cluster_size(n)
    ms = min_samples if min_samples is not None else HDBSCAN_MIN_SAMPLES

    ks = []
    for semilla in semillas:
        labels = _hdbscan_labels_con_semilla(embeddings, mcs, ms, semilla, n_componentes_quitar)
        ks.append(len(set(labels) - {-1}))

    ks = np.array(ks)
    valores, conteos = np.unique(ks, return_counts=True)
    moda = valores[np.argmax(conteos)]
    pct_moda = conteos.max() / len(ks) * 100

    return {
        "modelo": modelo,
        "palabra": palabra, "n_ocurrencias": n,
        "k_por_semilla": ks.tolist(),
        "k_moda": moda, "pct_semillas_en_moda": pct_moda,
        "k_min": ks.min(), "k_max": ks.max(), "k_std": ks.std(),
    }


# Ejemplo: correr sobre las palabras de calibración de un modelo
resultados = []
for modelo in MODELOS:
    print('-'*50)
    print(modelo)
    
    for palabra, embs in embeddings_calib_por_modelo[modelo].items():
        r = medir_estabilidad(modelo, palabra, embs)
        resultados.append(r)
        print(f"{palabra}: k por semilla = {r['k_por_semilla']}  "
              f"(moda={r['k_moda']}, {r['pct_semillas_en_moda']:.0f}% de las semillas, "
              f"rango [{r['k_min']}, {r['k_max']}])")
    
df = pd.DataFrame(resultados)
df.to_csv("estabilidad_semillas.csv", index=False)
print("\nGuardado: estabilidad_semillas.csv")
print("\nCorrelación entre n_ocurrencias y estabilidad (pct_semillas_en_moda):")
print(df[["n_ocurrencias", "pct_semillas_en_moda"]].corr())

In [88]:
# %% [markdown]
# ### Inspección caso por caso (opcional)

# %%
# Ejemplo — ajusta modelo, palabra y parámetros elegidos:
# inspeccionar_ocurrencias(
#     "diciembre",
#     embeddings_calib_por_modelo["beto"],
#     tweets_calib_por_modelo["beto"],
#     min_cluster_size=5, min_samples=HDBSCAN_MIN_SAMPLES, n_componentes_quitar=1,
# )

# %% [markdown]
# ## 4. Orquestación — correr los 5 modelos sobre el corpus completo
#
# `N_COMPONENTES_QUITAR_FINAL = 0`: el whitening no demostró distinguir
# separaciones reales de espurias (sección 3), así que no se usa una
# corrección sin evidencia de que funcione — se mantiene el pipeline simple.
# `presenta_multiples_clusters` describe estructura de embeddings observada,
# no polisemia validada (ver limitación en el encabezado del notebook).
# Guarda incrementalmente (append + flush por palabra) en un CSV nuevo, y es
# reanudable si el proceso se cae a mitad.

# %%
N_COMPONENTES_QUITAR_FINAL = 0    # sin evidencia de que el whitening ayude (sección 3) -- se deja en 0
RESULTADOS_CSV = "resultados_clustering_v2.csv"
CAMPOS_RESULTADOS = ["word_normalized", "modelo", "n_ocurrencias", "min_cluster_size_usado",
                      "n_componentes_quitar", "k_clusters", "pct_ruido", "silhouette", "pct_semillas_moda",
                      "presenta_multiples_clusters", "excluida_stopword"]
UMBRAL_RUIDO_NO_DETERMINADO = 50  # % de ruido a partir del cual el cluster no es confiable / no se reporta
 
 
def cargar_procesados(ruta=RESULTADOS_CSV):
    """Devuelve el set de (modelo, word_normalized) ya procesados, leyendo el
    CSV si existe. Tolera una última línea corrupta (ej. si el proceso se cayó
    a mitad de una escritura)."""
    if not os.path.exists(ruta):
        return set()
    try:
        df_previo = pd.read_csv(ruta, on_bad_lines="skip")
    except TypeError:  # pandas viejo sin on_bad_lines
        df_previo = pd.read_csv(ruta, error_bad_lines=False)
    return set(zip(df_previo["modelo"], df_previo["word_normalized"]))
 
 
def clusterizar_todas_las_palabras_resumible(nombre_modelo, min_cluster_size=None, min_samples=None,
                                               n_componentes_quitar=N_COMPONENTES_QUITAR_FINAL, ruta=RESULTADOS_CSV,
                                               semillas=SEMILLAS_ESTABILIDAD):
    """Igual que clusterizar_palabra, pero recorre TODAS las palabras del
    modelo y guarda el resultado de CADA una apenas termina (append + flush a
    `ruta`), saltándose las que ya estén en el archivo — así, si el proceso se
    cae a mitad, vuelves a correr esta misma función y retoma donde se quedó.
    Corre con 5 semillas por palabra (ver _hdbscan_multi_semilla) -- 5x más
    lento que una sola semilla, pero necesario: reproducibilidad con una sola
    semilla NO implica que el resultado sea estable (ver hallazgo de
    estabilidad en el protocolo)."""
    emb, w, tweet_ids = cargar_embeddings(nombre_modelo)
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    palabras_arr = np.array([d for d in w])
    palabras_unicas = np.unique(palabras_arr)
 
    procesados = cargar_procesados(ruta)
    escribir_header = not os.path.exists(ruta)
    ya_hechas = sum(1 for p in palabras_unicas if (nombre_modelo, p) in procesados)
    print(f"--- {nombre_modelo}: {ya_hechas}/{len(palabras_unicas)} palabras ya procesadas, retomando ---")
 
    with open(ruta, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CAMPOS_RESULTADOS)
        if escribir_header:
            writer.writeheader()
            f.flush()
 
        for i, palabra in enumerate(palabras_unicas):
            if (nombre_modelo, palabra) in procesados:
                continue
 
            if palabra in STOPWORDS_ES:
                writer.writerow({
                    "word_normalized": palabra, "modelo": nombre_modelo,
                    "n_ocurrencias": None, "min_cluster_size_usado": None,
                    "n_componentes_quitar": None, "k_clusters": None, "pct_ruido": None,
                    "silhouette": None, "pct_semillas_moda": None,
                    "presenta_multiples_clusters": None, "excluida_stopword": True,
                })
                f.flush()
                continue
 
            idx = np.where(palabras_arr == palabra)[0]
            embs_palabra = emb[idx]
            n = len(embs_palabra)
 
            if n < MIN_OCURRENCIAS_PARA_CLUSTER:
                k, pct_ruido, mcs_usado, silhouette, pct_semillas_moda, presenta_multiples_clusters = (
                    None, None, None, None, None, None
                )
            else:
                mcs = min_cluster_size if min_cluster_size is not None else calcular_min_cluster_size(n)
                ms = min_samples if min_samples is not None else HDBSCAN_MIN_SAMPLES
                resultado = _hdbscan_multi_semilla(embs_palabra, mcs, ms,
                                                     n_componentes_quitar=n_componentes_quitar, semillas=semillas)
                k = resultado["k_clusters"]
                pct_ruido = resultado["pct_ruido"]
                silhouette = resultado["silhouette"]
                pct_semillas_moda = resultado["pct_semillas_moda"]
                mcs_usado = mcs
                presenta_multiples_clusters = None if pct_ruido > UMBRAL_RUIDO_NO_DETERMINADO else (k > 1)
 
            writer.writerow({
                "word_normalized": palabra, "modelo": nombre_modelo,
                "n_ocurrencias": n, "min_cluster_size_usado": mcs_usado,
                "n_componentes_quitar": n_componentes_quitar if n >= MIN_OCURRENCIAS_PARA_CLUSTER else None,
                "k_clusters": k, "pct_ruido": pct_ruido, "silhouette": silhouette,
                "pct_semillas_moda": pct_semillas_moda,
                "presenta_multiples_clusters": presenta_multiples_clusters, "excluida_stopword": False,
            })
            f.flush()  # persiste en disco apenas termina esta palabra, no al final del loop
 
            if i % 500 == 0:
                print(f"  [{nombre_modelo}] {i}/{len(palabras_unicas)} palabras procesadas")
 
    print(f"--- {nombre_modelo}: completado ---")
 
 
for nombre_modelo in MODELOS:
    clusterizar_todas_las_palabras_resumible(nombre_modelo)
 
resultados_df = pd.read_csv(RESULTADOS_CSV)
print(resultados_df.head(20))
print(f"\nTotal de filas (palabra x modelo): {len(resultados_df)}")

--- mex_state: 0/12434 palabras ya procesadas, retomando ---
  [mex_state] 0/12434 palabras procesadas
  [mex_state] 500/12434 palabras procesadas
  [mex_state] 1000/12434 palabras procesadas
  [mex_state] 1500/12434 palabras procesadas
  [mex_state] 2000/12434 palabras procesadas
  [mex_state] 2500/12434 palabras procesadas
  [mex_state] 3000/12434 palabras procesadas
  [mex_state] 3500/12434 palabras procesadas
  [mex_state] 4000/12434 palabras procesadas
  [mex_state] 4500/12434 palabras procesadas
  [mex_state] 5000/12434 palabras procesadas
  [mex_state] 5500/12434 palabras procesadas
  [mex_state] 6000/12434 palabras procesadas
  [mex_state] 6500/12434 palabras procesadas
  [mex_state] 7000/12434 palabras procesadas
  [mex_state] 7500/12434 palabras procesadas
  [mex_state] 8000/12434 palabras procesadas
  [mex_state] 8500/12434 palabras procesadas
  [mex_state] 9000/12434 palabras procesadas
  [mex_state] 9500/12434 palabras procesadas
  [mex_state] 10000/12434 palabras procesad

In [35]:
1

1